In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff
from IPython.display import Image

In [40]:
df = pd.read_csv('df_all_clear.csv')
df.head()


,Цена,Дата публикации,Город,is_class_eco,is_class_comfort,is_class_business,is_class_elite,is_brick,is_monolith,is_panel,...,Площадь,Этаж,Этажность_дома,Цена_за_квадратный_метр,Rooms_Count,Property_Type_Квартира,Property_Type_Своб. планировка,Property_Type_Студия,Year_Public,Month_Public
0,5964400,2025-09-22 13:37:50,Киров,0,0,0,0,0,0,0,...,3.891820,17,17,11.730126,2,1,0,0,2025,9
1,5829810,2025-09-29 20:46:23,Киров,0,0,0,0,0,0,0,...,3.688879,11,12,11.914940,1,1,0,0,2025,9
2,6400900,2025-09-29 10:26:27,Киров,0,1,0,0,0,0,0,...,3.970292,17,17,11.720714,3,1,0,0,2025,9
3,5814900,2025-09-29 20:37:16,Киров,0,0,0,0,0,0,0,...,3.688879,10,12,11.912379,1,1,0,0,2025,9
4,6315750,2025-09-29 20:42:24,Киров,0,0,0,0,0,0,0,...,3.713572,12,12,11.969684,1,1,0,0,2025,9


In [41]:
df.shape

(18050, 58)

In [ ]:
# Построение гистограммы для распределения цен
fig_price_hist = px.histogram(
    df, 
    x='Цена', 
    nbins=100,
    title='Распределение Цен на Недвижимость',
    labels={'Цена': 'Цена (руб.)'},
    marginal='box' # Дополнительная информация о распределении
)

fig_price_hist.update_layout(
    title_x=0.5,
    xaxis_title='Цена (руб.)',
    yaxis_title='Количество Объектов'
)

fig_price_hist.show()



Распределение цен сильно смещено вправо. Это означает, что подавляющее большинство квартир находится в более низком ценовом сегменте. При этом существует "длинный хвост" из очень дорогих, элитных объектов, которые являются скорее исключением, чем правилом.

In [43]:
# Применение логарифмического преобразования
df['Цена_log'] = np.log1p(df['Цена'])

# Создание двух гистограмм для сравнения
fig = make_subplots(rows=1, cols=2, subplot_titles=('Исходное Распределение Цен', 'Логарифмированное Распределение Цен'))

# Исходное распределение
fig.add_trace(go.Histogram(x=df['Цена'], nbinsx=100, name='Цена'), row=1, col=1)

# Логарифмированное распределение
fig.add_trace(go.Histogram(x=df['Цена_log'], nbinsx=100, name='log(Цена + 1)'), row=1, col=2)

fig.update_layout(
    title_text='Сравнение Распределения Цены до и после Логарифмирования',
    title_x=0.5,
    showlegend=False
)
fig.update_xaxes(title_text='Цена (руб.)', row=1, col=1)
fig.update_xaxes(title_text='log(Цена + 1)', row=1, col=2)
fig.update_yaxes(title_text='Количество', row=1, col=1)

fig.show()



Логарифмирование переменной Цена нормализует распределение. Правая гистограмма выглядит гораздо более симметричной, похожей на нормальное распределение, по сравнению с сильно скошенной левой гистограммой. Это подтверждает, что для регрессионного моделирования цен лучше использовать логарифмированную версию.

In [44]:
# Визуализация распределения цены за квадратный метр
fig_price_sqm_hist = px.histogram(
    df, 
    x='Цена_за_квадратный_метр', 
    nbins=100,
    title='Распределение Цены за Квадратный Метр',
    labels={'Цена_за_квадратный_метр': 'Цена за м² (руб.)'},
    marginal='box'
)

fig_price_sqm_hist.update_layout(
    title_x=0.5,
    xaxis_title='Цена за м² (руб.)',
    yaxis_title='Количество Объектов'
)

fig_price_sqm_hist.show()



Данные неоднородны и представляют как минимум три разных рынка.

In [ ]:
# Группировка данных по субъектам РФ и расчет медианной цены за м²
regional_prices = df.groupby('Субъект РФ')['Цена_за_квадратный_метр'].median().sort_values(ascending=False).reset_index()

# Построение столбчатой диаграммы
fig_region_bar = px.bar(
    regional_prices,
    x='Субъект РФ',
    y='Цена_за_квадратный_метр',
    title='Медианная Цена за Квадратный Метр по Субъектам РФ',
    labels={'Субъект РФ': 'Субъект РФ', 'Цена_за_квадратный_метр': 'Медианная цена за м² (руб.)'}
)

fig_region_bar.update_layout(title_x=0.5)
fig_region_bar.show()




**Выводы:**
- Наблюдается  вариативность цен между субъектами РФ, что подтверждает региональную специфику рынка недвижимости.


**Гипотезы:**
- Высокие медианные цены могут быть связаны с уровнем экономического развития региона, наличием рабочих мест и развитой инфраструктурой.
- Субъекты с низкими ценами могут характеризоваться низкой покупательной способностью населения или недостаточным предложением качественного жилья.


In [ ]:
# Для наглядности ограничимся городами с достаточным количеством объявлений
city_counts = df['Город'].value_counts()
top_cities = city_counts[city_counts > 50].index 

df_top_cities = df[df['Город'].isin(top_cities)]

# Сортировка городов по медианной цене для более наглядного графика
sorted_cities = df_top_cities.groupby('Город')['Цена_за_квадратный_метр'].median().sort_values(ascending=False).index

# Построение box plot
fig_city_box = px.box(
    df_top_cities,
    x='Город',
    y='Цена_за_квадратный_метр',
    title='Распределение Цен за Квадратный Метр по Городам',
    labels={'Город': 'Город', 'Цена_за_квадратный_метр': 'Цена за м² (руб.)'},
    category_orders={'Город': sorted_cities} # Применяем сортировку
)

fig_city_box.update_layout(title_x=0.5)
fig_city_box.show()


**Выводы:**
- Даже в рамках одного субъекта РФ цены за м² сильно варьируются между городами, что указывает на локальную специфику спроса и предложения.
- В крупных городах разброс цен обычно больше, что отражает более сегментированный рынок (от эконом- до элитного класса).
- Наблюдается наличие выбросов в некоторых городах, что может указывать на объекты с уникальными характеристиками или ошибки в данных.

**Гипотезы:**
- Города с высокими медианными ценами могут быть центрами притяжения (рабочие места, образование, культура).
- Широкий разброс цен в некоторых городах может быть связан с неравномерным распределением объектов по районам с разной престижностью.


In [55]:

# Выбор числовых столбцов для анализа корреляции
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

# Расчет матрицы корреляции
corr_matrix = df[numeric_cols].corr()

# Для наглядности выберем только признаки с высокой корреляцией с ценой
corr_target = corr_matrix['Цена'].abs().sort_values(ascending=False)
top_corr_features = corr_target[corr_target > 0.2].index

# Тепловая карта для наиболее коррелирующих признаков
fig_heatmap = go.Figure(data=go.Heatmap(
                   z=df[top_corr_features].corr().values,
                   x=top_corr_features.tolist(),
                   y=top_corr_features.tolist(),
                   colorscale='RdBu',
                   zmin=-1,
                   zmax=1,
                   text=df[top_corr_features].corr().round(2).values,
                   texttemplate="%{text}"
))

fig_heatmap.update_layout(
    title='Тепловая Карта Корреляций Ключевых Числовых Признаков',
    title_x=0.5
)

fig_heatmap.show()


**Выводы:**
- Наблюдается положительная корреляция между признаками, которые логически связаны (например, площадь и цена).
- Признаки, коррелирующие с целевой переменной "Цена", могут быть важными предикторами в модели.
- Наличие сильных корреляций между признаками (мультиколлинеарность) требует внимания при построении модели, чтобы избежать избыточности признаков.

**Гипотезы:**
- Высокая корреляция между "Площадь" и "Цена" подтверждает интуитивное ожидание: большие квартиры стоят дороже.
- Корреляции между географическими признаками могут быть связаны с расположением в одних и тех же районах.


In [ ]:
# Диаграмма рассеяния для площади и цены
fig_scatter_area_price = px.scatter(
    df.sample(5000) if len(df) > 5000 else df, # Используем выборку для больших датасетов
    x='Площадь',
    y='Цена',
    color='Город', # Раскрашиваем точки по городам
    title='Зависимость Цены от Площади в Разрезе Городов',
    labels={'Площадь': 'Площадь (м²)', 'Цена': 'Цена (руб.)'},
    hover_data=['Цена_за_квадратный_метр']
)

fig_scatter_area_price.update_layout(title_x=0.5)
fig_scatter_area_price.show()


**Выводы:**
- Существует положительная зависимость между площадью квартиры и её ценой, что соответствует рыночной логике.
- Разные города имеют разные облака точек, что указывает на разную ценовую динамику по городам при одинаковой площади.
- В некоторых городах цена за м² выше, в других - ниже при одинаковой площади.

**Гипотезы:**
- Линейная или близкая к линейной зависимость предполагает, что площадь является сильным предиктором цены.
- Различия в расположении облаков по городам могут объясняться факторами: уровень жизни, местоположение, престиж района.
- Нелинейность на графике может указывать на элитный сегмент, где цена растет непропорционально площади.


In [ ]:
# Диаграмма рассеяния для расстояния до центра и цены за м²
fig_scatter_dist_price = px.scatter(
    df.sample(5000) if len(df) > 5000 else df,
    x='dist_to_city_center',
    y='Цена_за_квадратный_метр',
    color='is_class_business', 
    trendline='ols', 
    title='Зависимость Цены за м² от Расстояния до Центра Города',
    labels={'dist_to_city_center': 'Расстояние до центра (км)', 'Цена_за_квадратный_метр': 'Цена за м² (руб.)'}
)

fig_scatter_dist_price.update_layout(title_x=0.5)
fig_scatter_dist_price.show()


Чем дальше объект от центра, тем он, как правило, дешевле.
Точки, относящиеся к бизнес-классу, будут систематически располагаться выше (иметь более высокую цену за м²), чем объекты другого класса, даже на одинаковом удалении от центра.
Можем наблюдать выбросы, от коротых нужно избавиться.

In [ ]:
# Создаем временный DF для удобства визуализации
material_df = df.melt(
    id_vars=['Цена_за_квадратный_метр'], 
    value_vars=['is_brick', 'is_monolith', 'is_panel'],
    var_name='Материал', 
    value_name='Есть признак'
)
material_df = material_df[material_df['Есть признак'] == True]
material_df['Материал'] = material_df['Материал'].replace({'is_brick': 'Кирпич', 'is_monolith': 'Монолит', 'is_panel': 'Панель'})

# Box plot для материалов стен
fig_material = px.box(
    material_df, 
    x='Материал', 
    y='Цена_за_квадратный_метр',
    title='Влияние Материала Стен на Цену за м²',
    labels={'Материал': 'Материал стен', 'Цена_за_квадратный_метр': 'Цена за м² (руб.)'}
)
fig_material.update_layout(title_x=0.5)
fig_material.show()



**Выводы:**
- Разные материалы стен показывают различное распределение цен за м², что указывает на их влияние на стоимость недвижимости.
- Монолитные дома, как правило, имеют более высокую цену за м² по сравнению с панельными, что может быть связано с качеством строительства и престижностью.

**Гипотезы:**
- Монолит ассоциируется с современным строительством и лучшей звукоизоляцией, что оправдывает премию в цене.
- Кирпичные дома могут занимать промежуточное положение между монолитом и панелью по цене, что отражает баланс между качеством и стоимостью строительства.
- Панельные дома чаще находятся в более старых районах или относятся к эконом-классу, что объясняет их более низкую стоимость.


In [51]:
# Список бинарных признаков для анализа
amenities_features = [
    'is_class_comfort', 'is_class_business', 'is_class_elite',
    'has_parking_underground', 'is_closed_yard', 'has_concierge'
]

# Создаем subplots
fig = make_subplots(rows=2, cols=3, subplot_titles=amenities_features)

row, col = 1, 1
for feature in amenities_features:
    # Box plot для каждого признака
    fig.add_trace(go.Box(y=df[df[feature] == True]['Цена_за_квадратный_метр'], name='Есть'), row=row, col=col)
    fig.add_trace(go.Box(y=df[df[feature] == False]['Цена_за_квадратный_метр'], name='Нет'), row=row, col=col)
    
    col += 1
    if col > 3:
        col = 1
        row += 1

fig.update_layout(
    title_text='Влияние Класса и Удобств на Цену за м²',
    title_x=0.5,
    showlegend=False,
    height=700
)
fig.show()

Все перечисленные признаки ('Комфорт-класс', 'Бизнес-класс', 'Элитный-класс', 'Подземный паркинг', 'Закрытый двор', 'Консьерж') положительно влияют на цену за квадратный метр. То есть, для каждой пары 'Есть' / 'Нет', медианная цена и весь диапазон цен у группы 'Есть' будут выше, чем у группы 'Нет'.


In [ ]:
# Создание нового признака - относительное положение этажа
df['floor_ratio'] = (df['Этаж'] / df['Этажность_дома']).fillna(0.5) # Заполняем пропуски медианным значением

# Диаграмма рассеяния для относительного положения этажа
fig_floor_ratio = px.scatter(
    df.sample(5000) if len(df) > 5000 else df,
    x='floor_ratio',
    y='Цена_за_квадратный_метр',
    title='Зависимость Цены за м² от Относительного Положения Этажа',
    labels={'floor_ratio': 'Относительное положение этажа (Этаж / Этажность)', 'Цена_за_квадратный_метр': 'Цена за м² (руб.)'},
    trendline='lowess' # Используем сглаженную линию тренда 
)

fig_floor_ratio.update_layout(title_x=0.5)
fig_floor_ratio.show()



**Выводы:**
- Наблюдается зависимость между относительным положением этажа и ценой за м².
- Экстремальные значения (очень низкие или очень высокие этажи относительно этажности дома) могут влиять на цену по-разному.

**Гипотезы:**
- Средние и верхние этажи (floor_ratio > 0.5) могут стоить дороже из-за лучшего вида, меньшего шума и большего количества света.
- Первые этажи (floor_ratio близко к 0) могут быть дешевле из-за шума, меньшей приватности и безопасности.
- Самые верхние этажи могут иметь премию из-за видовых характеристик или, наоборот, быть дешевле из-за проблем с лифтом или инсоляцией.


In [53]:
# Агрегация данных на уровне города
city_agg = df.groupby('Город').agg(
    median_price_sqm=('Цена_за_квадратный_метр', 'median'),
    avg_salary=('Cредняя зп в городе, тыс руб (2025)', 'mean'),
    population=('2015 население', 'mean'),
    population_dynamics=('Динамика населения за 10 лет', 'mean')
).reset_index()

# Диаграмма рассеяния для средней зарплаты и медианной цены
fig_macro = px.scatter(
    city_agg,
    x='avg_salary',
    y='median_price_sqm',
    size='population',
    text='Город',
    title='Связь Средней Зарплаты и Медианной Цены за м² по Городам',
    labels={'avg_salary': 'Средняя ЗП в городе (тыс. руб.)', 'median_price_sqm': 'Медианная цена за м² (руб.)'}
)

fig_macro.update_traces(textposition='top center')
fig_macro.update_layout(title_x=0.5, height=600)
fig_macro.show()


**Выводы:**
- Города с высокими зарплатами обычно имеют более высокие цены на недвижимость, что отражает платежеспособный спрос.
- Размер точек (население) показывает, что крупные города могут иметь как высокие, так и относительно низкие цены, в зависимости от региона.

**Гипотезы:**
- Высокая средняя зарплата создает платежеспособный спрос, что толкает цены вверх через механизм рыночного равновесия.
- Города с растущим населением (положительная динамика) могут иметь более высокий рост цен на недвижимость из-за увеличения спроса.
- Региональные различия в ценах могут быть не только следствием зарплат, но и других факторов: доступность земли, стоимость строительства, инфраструктура, миграционные потоки.


## Вывод:

Цена на недвижимость (и общая, и за квадратный метр) является сложным показателем, который зависит от множества факторов. 

Данные показывают, что цены не распределены нормально, а имеют сильный скос вправо, что указывает на то, что большинство объектов недвижимости находится в низком и среднем ценовом сегменте, но существует "длинный хвост" очень дорогих, элитных объектов.


### 1. Распределение цен:

Распределение как общей цены, так и цены за квадратный метр, сильно смещено вправо. Для статистического моделирования целесообразно использовать логарифмированное значение цены (например, log(Цена + 1)), так как оно имеет распределение, более близкое к нормальному. 

### 2. Географические факторы (Регион и Город):

Наблюдается значительная вариативность цен за м² в зависимости от города, даже в пределах одного и того же субъекта РФ. Это указывает на сильное влияние локальных факторов спроса и предложения. 

Города с более высоким средним уровнем заработной платы, как правило, имеют и более высокие медианные цены на недвижимость. Это подтверждает прямую связь между платежеспособным спросом населения и стоимостью жилья. 

Размер города (население) также влияет на цены, но эта связь менее однозначна. Крупные города могут иметь как высокие, так и относительно низкие цены в зависимости от региональной экономики. 

#### Гипотезы: 

Города, являющиеся "центрами притяжения" (из-за наличия рабочих мест, образовательных учреждений, культурной жизни), имеют более высокие цены на недвижимость. 

Города с положительной динамикой роста населения, вероятно, будут демонстрировать и более высокий рост цен на жилье из-за растущего спроса. 

### 3. Характеристики недвижимости:

#### Площадь:

Существует сильная положительная корреляция между общей площадью квартиры и её итоговой ценой.

##### Гипотеза: 

Эта зависимость близка к линейной, что делает "Площадь" одним из ключевых предикторов при моделировании цены. 

#### Материал стен:

Монолитные дома в среднем имеют более высокую цену за м², чем панельные. 

##### Гипотеза: 

Это связано с тем, что "Монолит" ассоциируется с более современным и качественным строительством (например, лучшей звукоизоляцией), что и формирует ценовую премию. "Кирпич" занимает промежуточное положение. "Панельные" дома, вероятно, чаще относятся к эконом-классу или расположены в менее престижных районах. 

#### Класс жилья и удобства:

Наличие атрибутов, повышающих класс жилья (бизнес-класс, элитный класс, комфорт-класс), а также наличие подземного паркинга, закрытого двора или консьержа, стабильно повышает медианную цену за квадратный метр. 

#### Этажность:

Наблюдается зависимость цены за м² от относительного расположения этажа (отношение этажа квартиры к общей этажности дома). 

##### Гипотеза: 
Квартиры на первых этажах (низкий floor_ratio) дешевле из-за соображений безопасности и шума. Средние и верхние этажи (особенно в домах с хорошими видами) ценятся выше. 
